# PandaPickCube: пять экспериментов

Копия для повторения, подготовленная из рабочего ноутбука после экспериментов. Выводы очищены; результаты находятся в logs/ и reports/. Выберите GPU T4.


In [ ]:
%pip uninstall -y jax-cuda13-plugin jax-cuda13-pjrt

%pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126

%pip install "playground[learning]==0.1.0" "rsl-rl-lib==5.5.1" "torch==2.8.0" "jax[cuda12]==0.6.2" "mujoco==3.5.0" "mujoco-mjx==3.5.0" "numpy==2.2.6"

%pip install "tensorboard==2.21.0" "imageio[ffmpeg]==2.37.3"
%pip install "tensorboard==2.21.0" "imageio[ffmpeg]==2.37.3"


После полного завершения установки: Среда выполнения → Перезапустить сеанс. Затем выполняйте ячейки ниже, не повторяя установку. Не удаляйте среду.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
assert Path("/content/drive/MyDrive").is_dir()
p = Path("/content/drive/MyDrive/physicalAI/logs")
p.mkdir(parents=True, exist_ok=True)
(p / "write_test.txt").write_text("storage OK")
assert (p / "write_test.txt").read_text() == "storage OK"
print("Папка сохранения:", p)


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import torch
import jax
import mujoco
from importlib.metadata import version
from mujoco_playground import registry, wrapper_torch
from rsl_rl.runners import OnPolicyRunner

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
print("JAX:", jax.__version__)
print("Устройства JAX:", jax.devices())
print("MuJoCo:", mujoco.__version__)
print("RSL-RL:", version("rsl-rl-lib"))

Выберите один эксперимент. Между запусками рекомендуется перезапустить сеанс, затем повторить подключение Диска и импорты. Обучение начинается с нуля. Существующие чекпоинты защищены от перезаписи.

In [ ]:
EXPERIMENT = "gripper_box_2"
EXPERIMENTS = {
    "baseline_1024": (1001, {}),
    "max_iters_500": (500, {}),
    "box_target_strong": (1001, {"box_target": 40.0}),
    "qpos_strong": (1001, {"robot_target_qpos": 30.0}),
    "gripper_box_2": (1001, {"gripper_box": 2.0}),
}
assert EXPERIMENT in EXPERIMENTS


In [ ]:
from pathlib import Path
"""Встроенные config.py + train.py для Colab (без внешнего репозитория)."""

import json
import os
import time
from datetime import datetime
from types import SimpleNamespace

import jax
import torch
from mujoco_playground import registry, wrapper_torch
from mujoco_playground.config import manipulation_params
from rsl_rl.runners import OnPolicyRunner

jax.config.update("jax_default_matmul_precision", "highest")

# --- Параметры запуска (аналог аргументов CLI из train.py) ---
NUM_ENVS = 1024
MAX_ITERS = EXPERIMENTS[EXPERIMENT][0]
SAVE_INTERVAL = 50
SEED = 1
EXP_NAME = EXPERIMENT
RESUME = None  # путь к чекпоинту или None
DEVICE = "cuda:0"

# --- config.py ---
ENV_NAME = "PandaPickCube"

ACTOR_HIDDEN_DIMS = [512, 256, 128]
CRITIC_HIDDEN_DIMS = [512, 256, 128]
ACTIVATION = "elu"
INIT_NOISE_STD = 1.0


def build_runner_cfg(obs_size) -> dict:
    cfg = manipulation_params.rsl_rl_config(ENV_NAME).to_dict()
    _policy = cfg.pop("policy", {})
    cfg["actor"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": ACTOR_HIDDEN_DIMS,
        "activation": ACTIVATION,
        "distribution_cfg": {
            "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
            "init_std": INIT_NOISE_STD,
        },
    }
    cfg["critic"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": CRITIC_HIDDEN_DIMS,
        "activation": ACTIVATION,
    }
    if isinstance(obs_size, dict):
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["privileged_state"]}
    else:
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["state"]}
    return cfg


def resolve_log_dir(exp_name, resume) -> str:
    if exp_name:
        name = exp_name
    elif resume:
        name = os.path.basename(os.path.dirname(os.path.abspath(resume)))
    else:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = f"{ENV_NAME}_{timestamp}"
    log_dir = os.path.join("/content/drive/MyDrive/physicalAI/logs", name)
    if any(Path(log_dir).glob("model_*.pt")):
        raise RuntimeError("В этой папке уже есть чекпоинты. Сохраните их и выберите другую папку до нового запуска.")
    os.makedirs(log_dir, exist_ok=True)
    return log_dir


args = SimpleNamespace(
    exp_name=EXP_NAME,
    num_envs=NUM_ENVS,
    max_iters=MAX_ITERS,
    device=DEVICE,
    resume=RESUME,
    save_interval=SAVE_INTERVAL,
    seed=SEED,
)

device = args.device
device_rank = int(device.split(":")[-1]) if "cuda" in device else 0

log_dir = resolve_log_dir(args.exp_name, args.resume)

print(f"Среда:            {ENV_NAME}")
print(f"Устройство:       {device}")
print(f"Параллельных сред: {args.num_envs}")
print(f"Итерации:         {args.max_iters}")
print(f"Лог/чекпоинты:    {log_dir}")
print()

env_cfg = registry.get_default_config(ENV_NAME)
env_cfg.sim_dt = 0.0025
for key, value in EXPERIMENTS[EXPERIMENT][1].items():
    env_cfg.reward_config.scales[key] = value
raw_env = registry.load(ENV_NAME, config=env_cfg)

brax_env = wrapper_torch.RSLRLBraxWrapper(
    raw_env,
    num_actors=args.num_envs,
    seed=args.seed,
    episode_length=env_cfg.episode_length,
    action_repeat=1,
    device_rank=device_rank,
)
brax_env.cfg = {}

cfg_dict = build_runner_cfg(raw_env.observation_size)
cfg_dict["seed"] = args.seed
cfg_dict["max_iterations"] = args.max_iters
cfg_dict["save_interval"] = args.save_interval

config_path = os.path.join(log_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(cfg_dict, f, indent=2, default=str)
print(f"Конфиг сохранён: {config_path}")

with open(os.path.join(log_dir, "env_config.json"), "w") as f:
    json.dump(env_cfg.to_dict(), f, indent=2, default=str)

print("learning_rate:", cfg_dict["algorithm"]["learning_rate"])
print("Параметры среды:", env_cfg)
runner = OnPolicyRunner(brax_env, cfg_dict, log_dir, device=device)

if args.resume:
    print(f"Загрузка чекпоинта: {args.resume}")
    runner.load(args.resume)

print("=" * 60)
print("  Начинаю обучение...")
print("=" * 60)
print()

t_start = time.time()
runner.learn(
    num_learning_iterations=args.max_iters,
    init_at_random_ep_len=False,
)
elapsed = time.time() - t_start

print()
print("=" * 60)
print(f"  Обучение завершено за {elapsed:.0f} с ({elapsed / 60:.1f} мин)")
print(f"  Логи и чекпоинты: {log_dir}")
print("=" * 60)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/physicalAI/logs
